# LLM Quantization Toolkit

This notebook implements **GPTQ**: Advanced quantization with sequential layer-wise optimization  

## Target Model
- **Model**: OpenLLaMA 7B
- **Output**: Quantized models at 4-bit and 8-bit precision
- **Calibration**: WikiText-2 dataset (128 samples)

## 1. Imports and Configuration

In [15]:
# ============================================================================
# IMPORTS AND CONFIGURATION
# ============================================================================

# Standard library imports
from pathlib import Path
from typing import Iterable, List

# PyTorch and deep learning
import torch
import torch.nn as nn

# Hugging Face transformers and datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

# Project imports
import utils.model_loader as model_loader
import utils.dataloader as dataloader

# ============================================================================
# CONFIGURATION
# ============================================================================

# Model configuration
MODEL_ID = model_loader.MODEL_ID

# Output configuration  
OUTPUT_ROOT = model_loader.OUTPUT_ROOT

# Calibration configuration
MIN_CALIB_SAMPLES = 32  # Minimum number of calibration samples for quantization

In [16]:
# Check if cache is on the mounted drive
from huggingface_hub.constants import HF_HUB_CACHE
print(f"Hugging Face Hub is pointing to: {HF_HUB_CACHE}")

Hugging Face Hub is pointing to: /mnt/huggingface_cache/hub


## 2. Calibration Data Loading

Calibration data is crucial for quantization algorithms (especially GPTQ and AWQ) to determine optimal quantization parameters. We use WikiText-2, a standard language modeling dataset with diverse, high-quality text.

In [17]:
# Load calibration data (this runs when the cell executes)
CALIB_TEXTS = dataloader.load_wikitext_calibration(num_samples=MIN_CALIB_SAMPLES)

Loading WikiText-2 dataset for calibration...
✓ Loaded 32 calibration samples from WikiText-2


## 3. GPTQ (Generative Pre-trained Transformer Quantization)


In [ ]:
# ============================================================================
# GPTQ (GENERATIVE PRE-TRAINED TRANSFORMER QUANTIZATION) IMPLEMENTATION
# ============================================================================

def _build_gptq_examples(tokenizer, texts: List[str], max_len: int = 256):
    """
    Prepare calibration examples for GPTQ quantization.
    
    Args:
        tokenizer: Tokenizer instance
        texts: List of calibration text strings
        max_len: Maximum sequence length
    
    Returns:
        List of tokenized examples (dicts with input_ids and attention_mask)
    """
    examples = []
    for t in texts:
        # Tokenize each text
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=max_len)
        examples.append({
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
        })
    return examples

# autoq version
def quantize_and_save_gptq(
    model_id: str,
    bits_list: Iterable[int],
    out_root: Path = OUTPUT_ROOT,
    calib_texts: List[str] = None,
    group_size: int = 128,
):
    """
    Quantize model using GPTQ at multiple bit widths and save results.
    
    GPTQ requires calibration data and performs layer-wise optimization
    to minimize quantization error.
    
    Args:
        model_id: HuggingFace model ID or path
        bits_list: List of bit widths to quantize to (e.g., [4])
        out_root: Root directory for saving quantized models
        calib_texts: Calibration texts (uses CALIB_TEXTS if None)
        group_size: Group size for quantization
    """
    from auto_gptq import AutoGPTQForCausalLM, BaseQuantizeConfig

    # Use default calibration texts if not provided
    calib_texts = calib_texts or CALIB_TEXTS
    
    # Load tokenizer and prepare examples
    tok = model_loader.load_tokenizer(model_id)
    examples = _build_gptq_examples(tok, calib_texts)

    # Quantize at each bit width
    for bits in bits_list:
        print(f"\n{'='*60}")
        print(f"Quantizing with GPTQ at {bits}-bit...")
        print(f"This may take several minutes...")
        print(f"{'='*60}")
        
        # Configure GPTQ quantization
        quantize_config = BaseQuantizeConfig(
            bits=bits,  # Target bit width
            group_size=group_size,  # Group size for quantization
            desc_act=False,  # Disable activation order optimization (faster)
        )

        # Load model with GPTQ wrapper
        model = AutoGPTQForCausalLM.from_pretrained(
            model_id,
            quantize_config=quantize_config,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map="auto",
        )

        # Perform GPTQ quantization (this is where optimization happens)
        print("Running GPTQ optimization...")
        model.quantize(examples)

        # Save quantized model
        out_dir = out_root / f"gptq_w{bits}"
        out_dir.mkdir(parents=True, exist_ok=True)
        model.save_quantized(out_dir, use_safetensors=True)
        tok.save_pretrained(out_dir)
        
        print(f"✓ Saved GPTQ {bits}-bit model to {out_dir}")

        # Clean up
        del model
        torch.cuda.empty_cache()



: 

## 4. Run Quantization


In [ ]:
# ============================================================================
# EXECUTE QUANTIZATION
# ============================================================================

if __name__ == "__main__":
    # Create output directory
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "="*70)
    print("STARTING QUANTIZATION PIPELINE")
    print("="*70)
    print(f"Model: {MODEL_ID}")
    print(f"Output directory: {OUTPUT_ROOT}")
    print(f"Calibration samples: {len(CALIB_TEXTS)}")
    print("="*70 + "\n")

    # GPTQ Quantization (High accuracy, slower)
    print("\n📊 STEP 2/3: GPTQ Quantization")
    quantize_and_save_gptq(
        MODEL_ID, 
        bits_list=[4],  # 4-bit GPTQ
        out_root=OUTPUT_ROOT
    )


STARTING QUANTIZATION PIPELINE
Model: openlm-research/open_llama_3b
Output directory: quantized_models_3b
Calibration samples: 32


📊 STEP 2/3: GPTQ Quantization


Using pad_token, but it is not set yet.



Quantizing with GPTQ at 4-bit...
This may take several minutes...
